# Syon + Gemma 2 FULL — Kaggle (5 fases + conversacao PT)

**Settings:**
- Accelerator: **GPU T4 x2** (ou P100)
- Internet: **ON**
- Persistence: **ON** (recomendado)

**Add Data (obrigatorio):**
1. `google/gemma-2-2b-it` (Kaggle Models)
2. Dataset `regyfelipe/syon-project` (codigo Syon)
3. Dataset `regyfelipe/syon-gemma-sft` (160k amostras SFT ja prontas)
4. Dataset `regyfelipe/syon-conversation-pt` (150k dialogos, opcional se ja no SFT)

**Output:** `/kaggle/working/syon-output/syon-gemma-full/syon-gemma-full`

In [ ]:
import os, sys, shutil
from pathlib import Path

SYON_SRC = Path("/kaggle/input/datasets/regyfelipe/syon-project/Syon")
PROJECT = Path("/kaggle/working/syon")

def ok(p):
    return p.is_dir() and (p / "training").is_dir() and (p / "scripts/kaggle/kaggle_gemma_full_train.py").exists()

if not ok(SYON_SRC):
    for script in Path("/kaggle/input").rglob("kaggle_gemma_full_train.py"):
        root = script.parent.parent.parent
        if ok(root):
            SYON_SRC = root
            break

if not ok(SYON_SRC):
    raise FileNotFoundError(f"Syon nao encontrado em /kaggle/input. Add Data: syon-project")

print(f"Origem: {SYON_SRC}")
if PROJECT.exists():
    shutil.rmtree(PROJECT)
shutil.copytree(SYON_SRC, PROJECT)
os.chdir(PROJECT)
sys.path[:0] = [str(PROJECT), str(PROJECT / "src")]
print(f"OK -> {PROJECT}")

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    raise RuntimeError("Ative GPU em Settings -> Accelerator")

In [ ]:
!pip install -q transformers accelerate peft bitsandbytes datasets pyyaml safetensors pyarrow

In [ ]:
# Listar datasets em /kaggle/input
from pathlib import Path
for p in sorted(Path("/kaggle/input").rglob("*")):
    if p.is_file() and p.name in ("gemma_sft_train.jsonl", "instruct_aira_pt.jsonl", "config.json"):
        print(p)

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
!python scripts/kaggle/run_gemma_full_kaggle.py

In [ ]:
from pathlib import Path
import json

out = Path("/kaggle/working/syon-output/syon-gemma-full")
final = out / "syon-gemma-full"
print(f"Final existe: {final.exists()}")
if (out / "training_summary.json").exists():
    print(json.dumps(json.loads((out / "training_summary.json").read_text()), indent=2))
if final.exists():
    print("Arquivos:", [f.name for f in final.iterdir()][:10])

## Publicar como Model Kaggle (opcional)

Salve o output deste notebook e use `upload_kaggle_model.py` ou publique manualmente em **Models -> New Model** apontando para `/kaggle/working/syon-output/syon-gemma-full/syon-gemma-full`.